In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from getpass import getpass
GIT_TOKEN = getpass('Enter your GitHub Personal Access Token (ghp_...): ')
GIT_USERNAME = "navdeepsingh1609"  # <-- your GitHub username
REPO_NAME = "CSC2529-Fall-Project"
REPO_URL = f"https://{GIT_TOKEN}@github.com/{GIT_USERNAME}/{REPO_NAME}.git"

Enter your GitHub Personal Access Token (ghp_...): ··········


In [ ]:
# Clone repo
%cd /content
!git clone {REPO_URL}
%cd {REPO_NAME}
!git checkout model2
# Init MambaIR submodule
!git submodule update --init --force --recursive --remote

# Install deps
!pip install -q -r requirements.txt

/content
Cloning into 'CSC2529-Fall-Project'...
remote: Enumerating objects: 455, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 455 (delta 34), reused 46 (delta 17), pack-reused 382 (from 1)
Receiving objects: 100% (455/455), 183.19 MiB | 44.69 MiB/s, done.
Resolving deltas: 100% (184/184), done.
Updating files: 100% (35/35), done.
/content/CSC2529-Fall-Project
Branch 'model2' set up to track remote branch 'model2' from 'origin'.
Switched to a new branch 'model2'
Submodule 'models/external/MambaIR' (https://github.com/csguoh/MambaIR.git) registered for path 'models/external/MambaIR'
Cloning into '/content/CSC2529-Fall-Project/models/external/MambaIR'...
Submodule path 'models/external/MambaIR': checked out '863ca48768a0a6da636bb354f585ef7cfa3fd558'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/

In [ ]:
!mkdir -p /content/dataset
!ln -s "/content/drive/MyDrive/Computational Imaging Project/Datasets/UDC-SIT/UDC-SIT" "/content/dataset/UDC-SIT"

In [ ]:
# Create subset
!python scripts/create_subset.py
!ls data/UDC-SIT_subset/train/input | head
!ls data/UDC-SIT_subset/val/input | head

Creating training subset...
Copying 30 files from /content/drive/MyDrive/Computational Imaging Project/UDC-SIT/UDC-SIT/training/input...
Creating validation subset...
Copying 7 files from /content/drive/MyDrive/Computational Imaging Project/UDC-SIT/UDC-SIT/validation/input...
Subset creation complete.
1000.npy
1001.npy
1002.npy
1003.npy
1008.npy
1009.npy
100.npy
1010.npy
1011.npy
1012.npy
1004.npy
1027.npy
1034.npy
1081.npy
1127.npy
1132.npy
114.npy


In [ ]:
# Quick Teacher training (small subset, for debugging)
!python train_teacher.py \
  --data-root /content/dataset/UDC-SIT \
  --preset quick \
  --save-loss-history \
  --drive-checkpoint-dir "/content/drive/MyDrive/Computational Imaging Project/checkpoints_quick"

--- [train_teacher_quick] Setting up system path...
--- [train_teacher_quick] Added /content/CSC2529-Fall-Project/models/external/MambaIR to sys.path
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

--- [train_teacher_quick] Configuration ---
Train Dir: data/UDC-SIT_subset/train
Val Dir:   data/UDC-SIT_subset/val
Patch Size: 256, Batch Size: 2
Epochs: 10, Learning Rate: 0.0001
Device: cuda
Checkpoint: teacher_quick_1epoch.pth
-----------------------------------

--- [train_teacher_quick] Loading datasets...
--- [UDCDataset] Loaded 30 files from data/UDC-SIT_subset/train
--- [UDCDataset] Loaded 7 files from data/UDC-SIT_subset/val
--- [train_teacher_quick] Initializing model...
--- [Teacher] Initializing with 4 in-channels and 4 out-channels.
--- [FrequencyDomainB

In [ ]:
# Quick Student Training KD (using the quick teacher)
!python train_student_kd.py \
  --data-root /content/dataset/UDC-SIT \
  --preset quick \
  --teacher-weights checkpoints_quick/teacher_4ch_quick_final.pth \
  --save-loss-history \
  --drive-checkpoint-dir "/content/drive/MyDrive/Computational Imaging Project/checkpoints_quick"

--- [train_student_kd_quick] Setting up system path...
--- [train_student_kd_quick] Added /content/CSC2529-Fall-Project/models/external/MambaIR to sys.path
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

--- [train_student_kd_quick] Configuration ---
Train Dir: data/UDC-SIT_subset/train
Val Dir:   data/UDC-SIT_subset/val
Patch Size: 256, Batch Size: 16
Epochs: 12, Learning Rate: 0.0002
Device: cuda
Teacher Weights: teacher_quick_1epoch.pth
Student Save Path: student_quick_1epoch.pth
-----------------------------------

--- [train_student_kd_quick] Loading datasets...
--- [UDCDataset] Loaded 30 files from data/UDC-SIT_subset/train
--- [UDCDataset] Loaded 7 files from data/UDC-SIT_subset/val
--- [train_student_kd_quick] Loading teacher from teacher_quick_1epoch.pt

In [ ]:
# Quick patch-based evaluation
!python testing_quick.py

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
--- [testing_quick] Split: val, Num images: 8
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.w

In [ ]:
# Copy dataset tarballs to local SSD
!mkdir -p /content/dataset
!cp "/content/drive/MyDrive/Computational Imaging Project/Datasets/UDC-SIT/UDC-SIT"-*.tar.gz /content/dataset/

# Extract training/validation
%cd /content/dataset
!cat UDC-SIT-*.tar.gz | tar -xzvf -
%cd /content/CSC2529-Fall-Project

/content/dataset
UDC-SIT/
UDC-SIT/validation/
UDC-SIT/validation/GT/
UDC-SIT/validation/GT/1257.npy
UDC-SIT/validation/GT/2491.npy
UDC-SIT/validation/GT/1398.npy
UDC-SIT/validation/GT/2889.npy
UDC-SIT/validation/GT/2585.npy
UDC-SIT/validation/GT/2561.npy
UDC-SIT/validation/GT/597.npy
UDC-SIT/validation/GT/2777.npy
UDC-SIT/validation/GT/210.npy
UDC-SIT/validation/GT/2581.npy
UDC-SIT/validation/GT/2783.npy
UDC-SIT/validation/GT/2762.npy
UDC-SIT/validation/GT/1330.npy
UDC-SIT/validation/GT/2716.npy
UDC-SIT/validation/GT/27.npy
UDC-SIT/validation/GT/2541.npy
UDC-SIT/validation/GT/418.npy
UDC-SIT/validation/GT/2537.npy
UDC-SIT/validation/GT/2582.npy
UDC-SIT/validation/GT/2276.npy
UDC-SIT/validation/GT/2639.npy
UDC-SIT/validation/GT/1613.npy
UDC-SIT/validation/GT/2587.npy
UDC-SIT/validation/GT/2590.npy
UDC-SIT/validation/GT/1334.npy
UDC-SIT/validation/GT/2511.npy
UDC-SIT/validation/GT/91.npy
UDC-SIT/validation/GT/2547.npy
UDC-SIT/validation/GT/2562.npy
UDC-SIT/validation/GT/2704.npy
UDC-SIT/

In [ ]:
# Full Teacher training (long run on full train split)
!python train_teacher.py \
  --data-root /content/dataset/UDC-SIT \
  --preset full \
  --save-loss-history \
  --drive-checkpoint-dir "/content/drive/MyDrive/Computational Imaging Project/checkpoints_full"

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
usage: train_teacher.py [-h] [--train-dir TRAIN_DIR] [--val-dir VAL_DIR]
                        [--patch-size PATCH_SIZE]
                        [--max-train-images MAX_TRAIN_IMAGES]
                        [--max-val-images MAX_VAL_IMAGES]
                        [--batch-size BATCH_SIZE] [--num-epochs NUM_EPOCHS]
                        [--learning-rate LEARNING_RATE]
                        [--num-workers NUM_WORKERS]
                        [--checkpoint-prefix CHECKPOINT_PREFIX]
                        [--drive-checkpoint-dir DRIVE_CHECKPOINT_DIR]
                        [--preset {full,quick}]
train_teacher.py: error: unrecognized arguments: --data-root /content/dataset/UDC-SIT --save-loss-history


In [ ]:
# Full Student Training KD (using the full teacher)
!python train_student_kd.py \
  --data-root /content/dataset/UDC-SIT \
  --preset full \
  --teacher-weights checkpoints_full/teacher_4ch_full_final.pth \
  --save-loss-history \
  --drive-checkpoint-dir "/content/drive/MyDrive/Computational Imaging Project/checkpoints_full"

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
usage: train_student_kd.py [-h] [--train-dir TRAIN_DIR] [--val-dir VAL_DIR]
                           [--patch-size PATCH_SIZE]
                           [--max-train-images MAX_TRAIN_IMAGES]
                           [--max-val-images MAX_VAL_IMAGES]
                           [--teacher-weights TEACHER_WEIGHTS]
                           [--batch-size BATCH_SIZE] [--num-epochs NUM_EPOCHS]
                           [--learning-rate LEARNING_RATE]
                           [--num-workers NUM_WORKERS]
                           [--checkpoint-prefix CHECKPOINT_PREFIX]
                           [--drive-checkpoint-dir DRIVE_CHECKPOINT_DIR]
                           [--preset {full,quick}]
train_student_kd.py: er

In [ ]:
#Testing on full image
!python testing_udc.py \
  --data-root /content/dataset/UDC-SIT/UDC-SIT \
  --split testing \
  --teacher-ckpt teacher_4ch_22epochs_bs8.pth \
  --student-ckpt student_distilled_4ch_full_data.pth \
  --patch-size 256 \
  --patch-batch 8 \
  --eval-mode full \
  --results-name full_run_phase \
  --num-plot-examples 5

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[testing_udc] Drive results root equals local results root; skipping Drive copy.

--- [testing_udc] Configuration ---
Data root:      /content/dataset/UDC-SIT/UDC-SIT
Split:          testing
Patch size:     256
Patch batch:    8
Max images:     None
Teacher ckpt:   teacher_4ch_22epochs_bs8.pth
Student ckpt:   student_distilled_4ch_full_data.pth
Results root:   /content/drive/MyDrive/Computational Imaging Project/Results/Model2/full_run_phase
Drive results:  None
Plot examples:  5
Eval mode:      full
Use tiling:     False
Save NPY:       True
Device:         cuda
-----------------------------------

Setting up LPIPS (VGG) on device: cuda
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/usr/loc

In [ ]:
#Eval for on full image
!python eval_srgb_udc.py \
  --data-root /content/dataset/UDC-SIT/UDC-SIT \
  --split testing \
  --teacher-pred-dir "/content/drive/MyDrive/Computational Imaging Project/Results/Model2/full_run/teacher_full_testing/npy" \
  --student-pred-dir "/content/drive/MyDrive/Computational Imaging Project/Results/Model2/full_run/student_full_testing/npy" \
  --results-name full_run_phase

[eval_srgb_udc] Drive results root equals local results root; skipping Drive copy.
Traceback (most recent call last):
  File "/content/CSC2529-Fall-Project/eval_srgb_udc.py", line 459, in <module>
    main()
  File "/content/CSC2529-Fall-Project/eval_srgb_udc.py", line 228, in main
    raise FileNotFoundError(f"[eval_srgb_udc] Missing input dir: {input_dir}")
FileNotFoundError: [eval_srgb_udc] Missing input dir: /content/dataset/UDC-SIT/UDC-SIT/testing/input


In [ ]:
#Viz for full image
!python viz_srgb_udc.py \
  --data-root /content/dataset/UDC-SIT/UDC-SIT \
  --split testing \
  --teacher-pred-dir "/content/drive/MyDrive/Computational Imaging Project/Results/Model2/full_run/teacher_full_testing/npy" \
  --student-pred-dir "/content/drive/MyDrive/Computational Imaging Project/Results/Model2/full_run/student_full_testing/npy" \
  --results-name full_run_phase \
  --max-images 238

[viz_srgb_udc] Drive results root equals local results root; skipping Drive copy.
Traceback (most recent call last):
  File "/content/CSC2529-Fall-Project/viz_srgb_udc.py", line 321, in <module>
    main()
  File "/content/CSC2529-Fall-Project/viz_srgb_udc.py", line 225, in main
    raise FileNotFoundError(f"[viz_srgb_udc] Missing input dir: {input_dir}")
FileNotFoundError: [viz_srgb_udc] Missing input dir: /content/dataset/UDC-SIT/UDC-SIT/testing/input


In [ ]:
#Testing on patch image
!python testing_udc.py \
  --data-root /content/dataset/UDC-SIT/UDC-SIT \
  --split testing \
  --teacher-ckpt teacher_4ch_22epochs_bs8.pth \
  --student-ckpt student_distilled_4ch_full_data.pth \
  --patch-size 256 \
  --eval-mode center_patch \
  --results-name patch_run \
  --num-plot-examples 5

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[testing_udc] Drive results root equals local results root; skipping Drive copy.

--- [testing_udc] Configuration ---
Data root:      /content/dataset/UDC-SIT/UDC-SIT
Split:          testing
Patch size:     256
Patch batch:    8
Max images:     None
Teacher ckpt:   teacher_4ch_22epochs_bs8.pth
Student ckpt:   student_distilled_4ch_full_data.pth
Results root:   /content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run
Drive results:  None
Plot examples:  5
Eval mode:      center_patch
Use tiling:     False
Save NPY:       True
Device:         cuda
-----------------------------------

Setting up LPIPS (VGG) on device: cuda
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/usr/

In [ ]:
#Eval on patch image
!python eval_srgb_udc.py \
  --data-root /content/dataset/UDC-SIT/UDC-SIT \
  --split testing \
  --teacher-pred-dir "/content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/teacher_full_testing/npy" \
  --student-pred-dir "/content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/student_full_testing/npy" \
  --results-name patch_run

[eval_srgb_udc] Drive results root equals local results root; skipping Drive copy.
=== [eval_srgb_udc] Config ===
Data root:      /content/dataset/UDC-SIT/UDC-SIT
Split:          testing
Teacher preds:  /content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/teacher_full_testing/npy
Student preds:  /content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/student_full_testing/npy
Max images:     None
Results root:   /content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run
Drive results:  None
WB mode:        none
Gamma:          1.0
[eval_srgb_udc] Saved CSV (pseudo_rgb) to /content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/testing_metrics_srgb_pseudo_rgb.csv
[eval_srgb_udc] Saved CSV (naive_rgb) to /content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/testing_metrics_srgb_naive_rgb.csv
[eval_srgb_udc] Saved CSV (naive_rgb_wbgamma) to /content/drive/MyDrive/Computational Im

In [ ]:
#Viz on patch image
!python viz_srgb_udc.py \
  --data-root /content/dataset/UDC-SIT/UDC-SIT \
  --split testing \
  --teacher-pred-dir "/content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/teacher_full_testing/npy" \
  --student-pred-dir "/content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/student_full_testing/npy" \
  --results-name patch_run \
  --max-images 238

[viz_srgb_udc] Drive results root equals local results root; skipping Drive copy.
=== [viz_srgb_udc] Config ===
Data root:      /content/dataset/UDC-SIT/UDC-SIT
Split:          testing
Teacher preds:  /content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/teacher_full_testing/npy
Student preds:  /content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/student_full_testing/npy
Max images:     238
Viz root:       /content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run
Drive viz root: None
WB mode:        none
Gamma:          1.0
[viz_srgb_udc] Saved panel: /content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/testing/1005_panel.png
[viz_srgb_udc] Saved panel: /content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/testing/1006_panel.png
[viz_srgb_udc] Saved panel: /content/drive/MyDrive/Computational Imaging Project/Results/Model2/patch_run/testing/1007_panel.png
[viz_srgb_udc]

In [ ]:
!git pull origin model2

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 523 bytes | 523.00 KiB/s, done.
From https://github.com/navdeepsingh1609/CSC2529-Fall-Project
 * branch            model2     -> FETCH_HEAD
   ef3757b..33e9383  model2     -> origin/model2
Updating ef3757b..33e9383
Fast-forward
 train_student_kd.py | 9 ++++++---
 train_teacher.py    | 9 ++++++---
 2 files changed, 12 insertions(+), 6 deletions(-)
